In [1]:
import torch

from tqdm import tqdm
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),           # (1, 28, 28)
    #transforms.Flatten(),            # (784)  
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=100, shuffle=False)

In [4]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.1307,), std=(0.3081,))
           )

In [6]:
train_data[0]

(tensor([[[-0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
           -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
           -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
           -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242],
          [-0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
           -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
           -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
           -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242],
          [-0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
           -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
           -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
           -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242],
          [-0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
           -0.4242, -0.4242, -0.424

In [8]:
sample_data, sample_label = train_data[0]
sample_data

tensor([[[-0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242],
         [-0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242],
         [-0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242],
         [-0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0

In [ ]:
class NnetMNISTclassificator(nn.Module):
    def __init__(self, inputs, outputs):
        super().__init__() # manda llamar el constructor superior
        
        self.flatten = nn.Flatten()
        self.input_layer = nn.Linear(inputs, 392)
        self.hidden1 = nn.Linear(392, outputs)
        self.activation_function = nn.functional.relu

    def forward(self, x):
        x = self.flatten(x)
        x = self.input_layer(x)
        x = self.activation_function(x)
        x = self.hidden1(x)
        x = self.activation_function(x)
        return x


In [ ]:
model = NnetMNISTclassificator(784, 10)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

epochs = 50
for epoch in tqdm(range(epochs)):
    model.train()
    for index, data in tqdm(enumerate(train_loader)):
        x_train, y_train = data
        
        optimizer.zero_grad()
        y_pred = model(x_train)
        
        loss = loss_function(y_pred, y_train)
        loss.backward()

        optimizer.step()



938it [00:12, 74.86it/s]:00<?, ?it/s]
938it [00:11, 79.12it/s]:12<00:50, 12.53s/it]
938it [00:11, 80.21it/s]:24<00:36, 12.14s/it]
938it [00:12, 75.12it/s]:36<00:23, 11.94s/it]
938it [00:11, 79.28it/s]:48<00:12, 12.15s/it]
100%|██████████| 5/5 [01:00<00:00, 12.08s/it]


In [14]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for data in tqdm(test_loader):
        x_test, y_test = data
        outputs = model(x_test)
        _, predicted = torch.max(outputs.data, 1)

        total += y_test.size(0)
        correct += (predicted == y_test).sum().item()

print(f"Accuracy: {correct / total}")

100%|██████████| 100/100 [00:01<00:00, 50.13it/s]

Accuracy: 0.9264


In [18]:
# Guardar modelo para inferencia posterior
torch.save(model.state_dict(), "models/NnetMNISTclassificator.pth")

In [19]:
# Cargar modelo para inferencia
model_loaded = NnetMNISTclassificator(784,10)
model_loaded.load_state_dict(torch.load("models/NnetMNISTclassificator.pth", 
                                        weights_only=True))
model.eval()

NnetMNISTclassificator(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (input_layer): Linear(in_features=784, out_features=10, bias=True)
)